> **Version corrigée** — ce notebook contient le code complet de tous les exercices, exécuté de bout en bout, ainsi qu'un élément de réponse pour chaque question d'observation. La version étudiant (à compléter soi-même) est téléchargeable depuis la page du cours.

# Régressions : linéaire, logistique et à noyau

**Notebook 6/9 — Introduction to Supervised Machine Learning**
*L3 → Master, Guillaume Metzler, Université Lyon 2*

Trois familles de modèles sous un même nom un peu trompeur : la régression
linéaire prédit une valeur réelle en ajustant une droite (ou un hyperplan)
aux données, la régression logistique prédit une probabilité
d'appartenance à une classe en faisant passer ce même hyperplan par une
sigmoïde, et la régression à noyau abandonne complètement l'idée d'une
forme fonctionnelle fixée à l'avance. On commence par la régression
linéaire, dont la solution s'obtient sans aucun algorithme itératif — un
cas assez rare dans ce cours pour être signalé.

- la **régression linéaire** : solution analytique (équations normales),
  puis versions régularisées (Ridge, Lasso) ;
- la **régression logistique**, binaire puis multinomiale (softmax) ;
- la **régression à noyau** (estimateur de Nadaraya-Watson), non
  paramétrique.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7, 4.5)


## 1. Régression linéaire

On dispose d'un échantillon $S=\{(x_i,y_i)\}_{i=1}^m \in \mathbb{R}^d\times\mathbb{R}$
et on suppose une relation

$$Y = \theta X + \varepsilon,$$

où $\varepsilon$ est le bruit du modèle. On cherche l'hypothèse linéaire

$$h(\theta, x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_d x_d,$$

en minimisant l'erreur quadratique (méthode des **moindres carrés**) :

$$\min_{\theta\in\mathbb{R}^{d+1}} \|y - h(\theta, X)\|_2^2
= \min_{\theta\in\mathbb{R}^{d+1}} \sum_{i=1}^m (y_i - h(\theta, x_i))^2.$$

Contrairement à presque tout le reste de ce cours, ce problème admet une
solution fermée, obtenue via les **équations normales** :

$$\hat\theta = (X^TX)^{-1}X^Ty,$$

où $X$ est la matrice de design (une colonne de 1 pour $\theta_0$). Pas de
descente de gradient, pas d'algorithme itératif : un seul produit matriciel
suffit.


In [ ]:
# Exemple travaillé : équations normales "à la main" vs scikit-learn
from sklearn.linear_model import LinearRegression

x = np.array([0.0, 1.0, 2.0, 3.0, 5.0])
y = np.array([1.0, 2.0, 2.0, 4.0, 5.0])

X_design = np.column_stack([np.ones_like(x), x])
theta_hat = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

reg = LinearRegression().fit(x.reshape(-1, 1), y)
theta_sklearn = np.array([reg.intercept_, reg.coef_[0]])

print("theta (équations normales) :", theta_hat)
print("theta (scikit-learn)       :", theta_sklearn)


### Exercice 1 (L3)

Faites la même chose sur l'échantillon vu en cours : $x=(1,2,3,4)$,
$y=(2,3,5,4)$.

1. Construisez la matrice de design $X\in\mathbb{R}^{4\times 2}$.
2. Calculez $\hat\theta=(X^TX)^{-1}X^Ty$ avec `numpy`
   (`np.linalg.solve` plutôt que `np.linalg.inv`, plus stable
   numériquement).
3. Comparez à `LinearRegression` de scikit-learn sur les mêmes données.
4. Calculez l'erreur quadratique moyenne du modèle obtenu sur cet
   échantillon.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([2.0, 3.0, 5.0, 4.0])

X_design = np.column_stack([np.ones_like(x), x])
theta_hat = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

reg = LinearRegression().fit(x.reshape(-1, 1), y)
theta_sklearn = np.array([reg.intercept_, reg.coef_[0]])

y_pred = X_design @ theta_hat
eqm = np.mean((y - y_pred) ** 2)

print(f"theta_hat (équations normales) : {theta_hat}")
print(f"theta (scikit-learn)           : {theta_sklearn}")
print(f"Erreur quadratique moyenne     : {eqm:.3f}")

assert np.allclose(theta_hat, theta_sklearn, atol=1e-8)
assert np.isclose(eqm, 0.45, atol=1e-2)
print("\nOK : theta_0 ~= 1.5, theta_1 ~= 0.8, EQM ~= 0.45.")


Ajustons maintenant une régression linéaire sur des données où la
relation sous-jacente n'est *pas* linéaire, et regardons ce que ça donne
du côté des résidus.

In [ ]:
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(0)
x_quad = np.sort(rng.uniform(-3, 3, size=100))
y_quad = 0.5 * x_quad ** 2 - x_quad + rng.normal(scale=1.0, size=100)

reg = LinearRegression().fit(x_quad.reshape(-1, 1), y_quad)
y_pred = reg.predict(x_quad.reshape(-1, 1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x_quad, y_quad, alpha=0.6, label="données observées")
axes[0].plot(x_quad, y_pred, color="crimson", lw=2, label="droite ajustée")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title("Régression linéaire sur une relation quadratique")
axes[0].legend()

residus = y_quad - y_pred
axes[1].scatter(x_quad, residus, alpha=0.6, color="darkorange")
axes[1].axhline(0, color="black", lw=1, linestyle="--")
axes[1].set_xlabel("x"); axes[1].set_ylabel("résidu")
axes[1].set_title("Résidus en fonction de x")
plt.tight_layout()
plt.show()

print(f"R^2 sur l'échantillon : {reg.score(x_quad.reshape(-1, 1), y_quad):.3f}")


$$ $$

**Question 1 :** Comment les résidus se répartissent-ils en fonction de x : au hasard autour de 0, ou selon une forme particulière ? Qu'est-ce que cela vous dit sur la pertinence d'un modèle linéaire ici ?

$$ $$

*Éléments de réponse.* Les résidus dessinent une forme en U : positifs aux deux extrémités, négatifs au centre. Ce n'est pas du bruit aléatoire mais un signal que le modèle n'a pas capturé — la relation sous-jacente est courbe, et une droite ne peut pas s'ajuster à une courbure. C'est un sous-apprentissage (biais élevé) : il faudrait soit ajouter des termes polynomiaux, soit utiliser un modèle non paramétrique comme la régression à noyau (section 3).

### Régularisation

Pour limiter le sur-apprentissage, on ajoute un terme de pénalité à la
perte :

$$\min_{\theta} \|y-h(\theta,X)\|_2^2 + \lambda \|\theta\|^2.$$

- **Ridge** (norme $\ell_2$, $\|\theta\|_2^2$) : rétrécit les
  coefficients vers 0, sans jamais les annuler exactement.
- **Lasso** (norme $\ell_1$, $\|\theta\|_1$) : induit de la
  **parcimonie** — certains coefficients sont mis exactement à 0, ce
  qui est utile pour sélectionner des variables en grande dimension.
  Inconvénient : la norme $\ell_1$ n'est pas différentiable en 0.

Plus $\lambda$ (noté `alpha` dans scikit-learn) est grand, plus le biais
augmente et la variance diminue.

In [ ]:
from sklearn.linear_model import Ridge

rng = np.random.RandomState(42)
n_samples, n_features = 100, 6
facteur_commun = rng.normal(size=n_samples)
X_corr = np.column_stack(
    [facteur_commun + 0.3 * rng.normal(size=n_samples) for _ in range(n_features)]
)
vrais_coefs = np.array([3.0, -2.0, 1.5, 0.0, 0.0, 0.5])
y_corr = X_corr @ vrais_coefs + rng.normal(scale=1.0, size=n_samples)

alphas = np.logspace(-2, 4, 60)
coefs_ridge = np.array([Ridge(alpha=a).fit(X_corr, y_corr).coef_ for a in alphas])

plt.figure(figsize=(7, 4.5))
for j in range(n_features):
    plt.plot(alphas, coefs_ridge[:, j], label=f"coef. variable {j + 1}")
plt.xscale("log")
plt.xlabel(r"$\alpha$ (échelle log)"); plt.ylabel("valeur du coefficient")
plt.title("Chemin de régularisation Ridge")
plt.axhline(0, color="black", lw=0.8)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


$$ $$

**Question 2 :** Que deviennent les coefficients quand alpha augmente ? L'un d'entre eux s'annule-t-il exactement, à un moment ou à un autre ?

$$ $$

*Éléments de réponse.* Tous les coefficients se rapprochent progressivement de 0 (rétrécissement), mais aucun ne l'atteint exactement, même pour des valeurs d'alpha très grandes : c'est la signature de la pénalité l2, qui contracte sans jamais forcer à zéro.

In [ ]:
from sklearn.linear_model import Lasso

alphas_lasso = np.logspace(-2, 1.5, 60)
coefs_lasso = np.array(
    [Lasso(alpha=a, max_iter=10000).fit(X_corr, y_corr).coef_ for a in alphas_lasso]
)

plt.figure(figsize=(7, 4.5))
for j in range(n_features):
    plt.plot(alphas_lasso, coefs_lasso[:, j], label=f"coef. variable {j + 1}")
plt.xscale("log")
plt.xlabel(r"$\alpha$ (échelle log)"); plt.ylabel("valeur du coefficient")
plt.title("Chemin de régularisation Lasso (mêmes données)")
plt.axhline(0, color="black", lw=0.8)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


$$ $$

**Question 3 :** Comparez à la figure précédente (Ridge) : que remarquez-vous cette fois pour certains coefficients à mesure qu'alpha augmente ?

$$ $$

*Éléments de réponse.* Plusieurs coefficients atteignent exactement 0 et y restent ensuite (les courbes se confondent avec l'axe horizontal) : Lasso élimine effectivement des variables du modèle. Ce sont d'ailleurs les coefficients associés aux deux variables non informatives de `vrais_coefs` (les deux 0.0) qui s'annulent parmi les premiers.

On balaie maintenant `alpha` en suivant l'erreur sur un jeu de train et
un jeu de test séparés, pour voir où se situe le compromis biais-variance
en pratique.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_reg, y_reg = make_regression(
    n_samples=150, n_features=20, n_informative=4, noise=8.0, random_state=1
)
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=1
)

alphas_sweep = np.logspace(-2, 3, 25)
mse_train, mse_test = [], []
for a in alphas_sweep:
    ridge = Ridge(alpha=a).fit(X_train, y_train)
    mse_train.append(mean_squared_error(y_train, ridge.predict(X_train)))
    mse_test.append(mean_squared_error(y_test, ridge.predict(X_test)))

plt.figure(figsize=(6.5, 4.5))
plt.plot(alphas_sweep, mse_train, marker="o", label="MSE train")
plt.plot(alphas_sweep, mse_test, marker="o", label="MSE test")
plt.xscale("log")
plt.xlabel(r"$\alpha$"); plt.ylabel("MSE")
plt.title("Régularisation Ridge : MSE train / test")
plt.legend()
plt.tight_layout()
plt.show()


### Exercice 2 (Master)

On reprend la même mécanique sur un jeu de données où seules 5 variables
sur 30 sont réellement informatives, en comparant cette fois trois
modèles.

1. Générez `make_regression(n_samples=200, n_features=30,
   n_informative=5, noise=10.0, random_state=0)`.
2. Séparez en train/test (`test_size=0.3`, `random_state=0`).
3. Entraînez `LinearRegression`, `Ridge(alpha=10.0)` et `Lasso(alpha=1.0)`
   sur le train.
4. Comparez leur MSE sur le test.
5. Comptez, pour le Lasso, le nombre de coefficients exactement nuls.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

X, y = make_regression(n_samples=200, n_features=30, n_informative=5,
                        noise=10.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

lin = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=10.0).fit(X_train, y_train)
lasso = Lasso(alpha=1.0).fit(X_train, y_train)

mse_lin = mean_squared_error(y_test, lin.predict(X_test))
mse_ridge = mean_squared_error(y_test, ridge.predict(X_test))
mse_lasso = mean_squared_error(y_test, lasso.predict(X_test))
n_coefs_nuls = int(np.sum(np.isclose(lasso.coef_, 0.0)))

print(f"MSE LinearRegression : {mse_lin:.2f}")
print(f"MSE Ridge            : {mse_ridge:.2f}")
print(f"MSE Lasso             : {mse_lasso:.2f}")
print(f"Coefficients Lasso mis à 0 : {n_coefs_nuls} / {X.shape[1]}")

assert n_coefs_nuls > 0, "Le Lasso devrait éliminer au moins quelques variables ici."
print("\nAvec seulement 5 variables informatives sur 30, le Lasso produit un")
print("modèle plus parcimonieux, ce qui limite le sur-apprentissage sur les")
print("variables non informatives, contrairement à la LinearRegression brute.")


## 2. Régression logistique

### Cas binaire

La régression logistique modélise, de façon linéaire, le logarithme du
rapport des chances (log-odds) d'appartenir à la classe 1 :

$$\ln\left(\frac{P(y=1\mid x)}{P(y=0\mid x)}\right) = h(w,b,x) = b + \langle x, w\rangle.$$

On en déduit la probabilité via la fonction **logistique (sigmoïde)** :

$$P(y=1\mid x) = \sigma(h(w,b,x)) = \frac{1}{1+\exp(-h(w,b,x))}.$$

Un exemple est (en général) prédit dans la classe 1 si
$P(y=1\mid x) > 0.5$, c'est-à-dire si $h(w,b,x) > 0$ : la frontière de
décision $\{x : h(w,b,x)=0\}$ est un hyperplan, même si la sortie du
modèle (une probabilité) ne l'est pas.

Les paramètres $(w,b)$ sont estimés par **maximum de vraisemblance**, ce
qui revient à minimiser la perte logistique (log-vraisemblance négative,
entropie croisée) :

$$\ell(w,b,S) = -\sum_{i=1}^m y_i \ln(g(w,b,x_i)) + (1-y_i)\ln(1-g(w,b,x_i)),
\qquad g(w,b,x) = \sigma(h(w,b,x)).$$

Contrairement à la régression linéaire, **il n'y a pas de solution
analytique** ici : le problème reste convexe, mais on le résout par un
algorithme itératif (descente de gradient, ou Newton-Raphson, qui utilise
la matrice hessienne pour converger plus vite).

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X_bin, y_bin = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.5, random_state=7,
)

clf = LogisticRegression().fit(X_bin, y_bin)

x_min, x_max = X_bin[:, 0].min() - 1, X_bin[:, 0].max() + 1
y_min, y_max = X_bin[:, 1].min() - 1, X_bin[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
proba = clf.predict_proba(np.column_stack([xx.ravel(), yy.ravel()]))[:, 1].reshape(xx.shape)

plt.figure(figsize=(6.5, 5.5))
contour = plt.contourf(xx, yy, proba, levels=20, cmap="RdBu_r", alpha=0.6)
plt.colorbar(contour, label=r"$P(y=1\mid x)$ prédite")
plt.contour(xx, yy, proba, levels=[0.5], colors="black", linewidths=2, linestyles="--")
plt.scatter(X_bin[y_bin == 0, 0], X_bin[y_bin == 0, 1], edgecolor="k", label="classe 0")
plt.scatter(X_bin[y_bin == 1, 0], X_bin[y_bin == 1, 1], edgecolor="k", marker="^", label="classe 1")
plt.xlabel("x1"); plt.ylabel("x2")
plt.title("Régression logistique : frontière (pointillés) et probabilité prédite")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Accuracy sur l'échantillon : {clf.score(X_bin, y_bin):.3f}")


$$ $$

**Question 4 :** Quelle est la forme de la frontière de décision (le trait pointillé) ? Est-ce cohérent avec l'expression de $h(w,b,x)$ ?

$$ $$

*Éléments de réponse.* La frontière est une droite (un hyperplan en dimension d) : elle correspond à h(w,b,x)=0, une équation linéaire en x. La sigmoïde transforme la sortie en probabilité, mais ne change pas la forme de la frontière — c'est toujours celle de la partie linéaire.

Voyons maintenant ce que devient cette frontière sur des données qui ne
sont clairement pas linéairement séparables.

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=200, noise=0.25, random_state=0)
clf_moons = LogisticRegression().fit(X_moons, y_moons)

x_min, x_max = X_moons[:, 0].min() - 1, X_moons[:, 0].max() + 1
y_min, y_max = X_moons[:, 1].min() - 1, X_moons[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
proba_moons = clf_moons.predict_proba(np.column_stack([xx.ravel(), yy.ravel()]))[:, 1].reshape(xx.shape)

plt.figure(figsize=(6.5, 5.5))
plt.contourf(xx, yy, proba_moons, levels=20, cmap="RdBu_r", alpha=0.6)
plt.contour(xx, yy, proba_moons, levels=[0.5], colors="black", linewidths=2, linestyles="--")
plt.scatter(X_moons[y_moons == 0, 0], X_moons[y_moons == 0, 1], edgecolor="k", label="classe 0")
plt.scatter(X_moons[y_moons == 1, 0], X_moons[y_moons == 1, 1], edgecolor="k", marker="^", label="classe 1")
plt.xlabel("x1"); plt.ylabel("x2")
plt.title("Régression logistique sur des données en croissants (make_moons)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Accuracy sur l'échantillon : {clf_moons.score(X_moons, y_moons):.3f}")


$$ $$

**Question 5 :** La frontière obtenue sépare-t-elle bien les deux croissants ? Quelle est, plus généralement, la limite du modèle logistique standard sur ce type de données ?

$$ $$

*Éléments de réponse.* Non : la frontière reste une droite, alors que la structure des données est courbe — une partie non négligeable des points est mal classée, et l'accuracy le confirme (nettement en dessous de celle obtenue sur le premier jeu de données). La régression logistique standard ne peut apprendre que des frontières linéaires ; il faudrait soit ajouter des caractéristiques non linéaires (x1^2, x1*x2, ...), soit changer de famille de modèle (SVM à noyau, arbres, ...).

In [ ]:
def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

scores = np.array([-2.0, -0.5, 0.0, 1.0, 3.0])
print("scores linéaires :", scores)
print("probabilités      :", np.round(sigmoide(scores), 3))


### Exercice 3 (L3)

Appliquons ceci à un vrai modèle entraîné, sans passer par
`predict_proba`.

1. Sur `X_bin, y_bin` (défini plus haut), entraînez une
   `LogisticRegression`.
2. Pour les 5 premiers exemples, calculez à la main
   $P(y=1\mid x) = \sigma(b + \langle w,x\rangle)$, en utilisant votre
   fonction `sigmoide` et les attributs `clf.coef_`, `clf.intercept_`.
3. Comparez à `clf.predict_proba(X)[:, 1]`.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

clf = LogisticRegression().fit(X_bin, y_bin)
X5 = X_bin[:5]

scores_lineaires = clf.intercept_[0] + X5 @ clf.coef_[0]
proba_manuelle = sigmoide(scores_lineaires)
proba_sklearn = clf.predict_proba(X5)[:, 1]

print("Probabilités (calcul manuel) :", np.round(proba_manuelle, 4))
print("Probabilités (scikit-learn)  :", np.round(proba_sklearn, 4))

assert np.allclose(proba_manuelle, proba_sklearn, atol=1e-8)
print("\nOK : le calcul manuel reproduit exactement predict_proba().")


### Cas multi-classes : régression logistique multinomiale

Pour $q>2$ classes, une approche naturelle serait d'apprendre $q-1$
modèles logistiques binaires par rapport à une classe de référence. De
façon équivalente, et indépendante du choix de cette référence, on
définit directement, pour chaque classe $k$, un vecteur $w^{(k)}$ et :

$$P(y=k\mid x) = \frac{\exp(\langle w^{(k)}, x\rangle)}{\sum_{l=1}^q \exp(\langle w^{(l)}, x\rangle)}$$

— la fonction **softmax**, une généralisation de la sigmoïde à $q$
classes (on vérifie que $\sum_k P(y=k\mid x)=1$). On prédit alors la
classe de plus forte probabilité,
$\hat y = \arg\max_{k\in\{1,\dots,q\}} P(y=k\mid x)$.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data[:, 2:4]  # longueur et largeur des pétales
y_iris = iris.target

clf_iris = LogisticRegression(max_iter=1000).fit(X_iris, y_iris)

x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = clf_iris.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

plt.figure(figsize=(6.5, 5.5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
for classe, nom in enumerate(iris.target_names):
    plt.scatter(X_iris[y_iris == classe, 0], X_iris[y_iris == classe, 1], label=nom, edgecolor="k")
plt.xlabel("longueur des pétales (cm)"); plt.ylabel("largeur des pétales (cm)")
plt.title("Régression logistique multinomiale (softmax) sur Iris")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Accuracy : {clf_iris.score(X_iris, y_iris):.3f}")
print(f"Nombre de classes : {len(iris.target_names)}, "
      f"forme de clf.coef_ : {clf_iris.coef_.shape}")


### Exercice 4 (Master)

On applique la même mécanique à un jeu de données à 13 variables (donc
impossible à visualiser directement), en ajoutant une étape de
normalisation.

1. Chargez `load_wine` (13 variables, 3 classes de vin).
2. Séparez en train/test (`test_size=0.3`, `random_state=0`,
   `stratify=y`).
3. Normalisez avec `StandardScaler`, puis entraînez une
   `LogisticRegression(max_iter=5000)` sur le train.
4. Affichez le `classification_report` sur le test.
5. Pour le premier exemple du test, vérifiez que les probabilités
   softmax somment bien à 1.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=5000).fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
print(classification_report(y_test, y_pred, target_names=wine.target_names))

proba_premier = clf.predict_proba(X_test_s[:1])
somme_proba = float(proba_premier.sum())
print("Probabilités du premier exemple test :", np.round(proba_premier[0], 4))
print("Somme des probabilités softmax        :", somme_proba)

assert np.isclose(somme_proba, 1.0, atol=1e-6)
print("\nOK : la somme des probabilités softmax vaut bien 1.")


## 3. Régression à noyau

Les deux modèles précédents supposent une forme fonctionnelle globale
fixe pour $h$. La régression à noyau change de paradigme : c'est une
méthode **non paramétrique**, où la prédiction en un point $x$ est une
moyenne pondérée des $y_i$ observés, pondération d'autant plus forte que
$x_i$ est proche de $x$. L'estimateur le plus classique est celui de
**Nadaraya-Watson** :

$$\hat m(x) = \frac{\sum_{i=1}^m K\left(\dfrac{x-x_i}{h}\right) y_i}
                    {\sum_{i=1}^m K\left(\dfrac{x-x_i}{h}\right)},$$

où $K$ est une fonction noyau (le noyau gaussien $K(u)=\exp(-u^2/2)$ est
le plus courant) et $h>0$ la **largeur de bande**, qui contrôle l'échelle
de voisinage prise en compte. Pas de $\theta$ à estimer ni d'algorithme
d'optimisation : tout l'échantillon d'apprentissage est gardé en mémoire
et réutilisé à chaque prédiction.

In [ ]:
def nadaraya_watson(x_train, y_train, x_query, bandwidth):
    x_train = np.asarray(x_train).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)
    x_query = np.asarray(x_query).reshape(-1)
    u = (x_query[:, None] - x_train[None, :]) / bandwidth
    poids = np.exp(-0.5 * u ** 2)  # noyau gaussien K(u) = exp(-u^2/2)
    numerateur = poids @ y_train
    denominateur = poids.sum(axis=1)
    denominateur = np.where(denominateur < 1e-12, 1e-12, denominateur)
    return numerateur / denominateur


rng = np.random.RandomState(0)
x_train = np.sort(rng.uniform(0, 10, size=60))
y_train = np.sin(x_train) + rng.normal(scale=0.3, size=60)
x_grid = np.linspace(0, 10, 300)

bandwidths = [0.05, 0.5, 3.0]
couleurs = ["crimson", "seagreen", "royalblue"]
etiquettes = ["h=0.05", "h=0.5", "h=3.0"]

plt.figure(figsize=(8, 5))
plt.scatter(x_train, y_train, s=20, color="gray", alpha=0.6, label="données bruitées")
plt.plot(x_grid, np.sin(x_grid), color="black", lw=1.5, linestyle=":", label="sin(x) (vraie fonction)")
for h, coul, etq in zip(bandwidths, couleurs, etiquettes):
    plt.plot(x_grid, nadaraya_watson(x_train, y_train, x_grid, h), color=coul, lw=2, label=etq)
plt.xlabel("x"); plt.ylabel("y")
plt.title("Estimateur de Nadaraya-Watson : effet de la largeur de bande h")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()


$$ $$

**Question 6 :** D'après le graphique, que se passe-t-il pour h=0.05 ? Et pour h=3.0 ? Laquelle des trois valeurs vous semble le meilleur compromis ?

$$ $$

*Éléments de réponse.* h=0.05 : la courbe colle presque parfaitement aux points bruités, elle zigzague — variance élevée, sur-apprentissage. h=3.0 : la courbe est presque plate, proche de la moyenne globale de y — biais élevé, sous-apprentissage. h=0.5 retrouve raisonnablement bien la sinusoïde sous-jacente : c'est le meilleur des trois compromis biais-variance ici.

Revenons à la comparaison amorcée en section 1 : sur ces mêmes données
non linéaires, que donne une régression linéaire face à la régression à
noyau ?

In [ ]:
reg_sin = LinearRegression().fit(x_train.reshape(-1, 1), y_train)
y_lin = reg_sin.predict(x_grid.reshape(-1, 1))
y_noyau = nadaraya_watson(x_train, y_train, x_grid, bandwidth=0.5)

plt.figure(figsize=(7.5, 4.5))
plt.scatter(x_train, y_train, s=20, color="gray", alpha=0.6, label="données")
plt.plot(x_grid, y_lin, color="royalblue", lw=2, label="régression linéaire")
plt.plot(x_grid, y_noyau, color="crimson", lw=2, label="régression à noyau (h=0.5)")
plt.plot(x_grid, np.sin(x_grid), color="black", lw=1, linestyle=":", label="sin(x)")
plt.legend()
plt.title("Régression linéaire vs régression à noyau, données non linéaires")
plt.tight_layout()
plt.show()

mse_lin = np.mean((y_train - reg_sin.predict(x_train.reshape(-1, 1))) ** 2)
mse_noyau = np.mean((y_train - nadaraya_watson(x_train, y_train, x_train, bandwidth=0.5)) ** 2)
print(f"MSE (train) régression linéaire : {mse_lin:.3f}")
print(f"MSE (train) régression à noyau  : {mse_noyau:.3f}")


$$ $$

**Question 7 :** Comparez les deux courbes ajustées. Que diriez-vous du modèle linéaire sur ces données : sous-apprentissage, sur-apprentissage, ou ajustement correct ?

$$ $$

*Éléments de réponse.* Le modèle linéaire est en sous-apprentissage : il ne peut représenter qu'une droite alors que la relation est sinusoïdale, et sa MSE d'entraînement reste nettement plus élevée. La régression à noyau, non paramétrique, épouse localement la forme de la sinusoïde et obtient une MSE bien plus faible pour une largeur de bande raisonnable — au prix de garder tout l'échantillon en mémoire plutôt qu'un vecteur de paramètres.

### Exercice 5 (L3)

On reprend le petit exemple du cours : 3 observations
$(x_1,y_1)=(1,10)$, $(x_2,y_2)=(2,20)$, $(x_3,y_3)=(4,30)$, un noyau
gaussien $K(u)=\exp(-u^2/2)$ et une largeur de bande $h=1$.

1. Utilisez `nadaraya_watson` (définie plus haut) pour prédire
   $\hat m(2)$.
2. Retrouvez ce résultat à la main : calculez les 3 poids
   $K((x_0-x_i)/h)$, normalisez-les (somme = 1), puis faites la moyenne
   pondérée des $y_i$.
3. Vérifiez que les deux résultats coïncident (à $10^{-6}$ près).

In [ ]:
import numpy as np

def nadaraya_watson(x_train, y_train, x_query, bandwidth):
    x_train = np.asarray(x_train).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)
    x_query = np.asarray(x_query).reshape(-1)
    u = (x_query[:, None] - x_train[None, :]) / bandwidth
    poids = np.exp(-0.5 * u ** 2)
    numerateur = poids @ y_train
    denominateur = poids.sum(axis=1)
    denominateur = np.where(denominateur < 1e-12, 1e-12, denominateur)
    return numerateur / denominateur

x_obs = np.array([1.0, 2.0, 4.0])
y_obs = np.array([10.0, 20.0, 30.0])
x0, h = 2.0, 1.0

m_hat_fonction = nadaraya_watson(x_obs, y_obs, [x0], h)[0]

u = (x0 - x_obs) / h
poids = np.exp(-0.5 * u ** 2)
poids_normalises = poids / poids.sum()
m_hat_manuel = np.sum(poids_normalises * y_obs)

print("Prédiction via nadaraya_watson :", round(float(m_hat_fonction), 3))
print("Prédiction calculée à la main  :", round(float(m_hat_manuel), 3))
print("Poids normalisés (x1, x2, x3)  :", np.round(poids_normalises, 3))

assert abs(m_hat_fonction - m_hat_manuel) < 1e-6
assert np.isclose(m_hat_fonction, 17.3, atol=0.2)
assert np.isclose(poids_normalises[1], 0.57, atol=0.02)
print("\nOK : les deux calculs coïncident (m_hat(2) ~= 17.3).")


Sur les données d'apprentissage elles-mêmes, la MSE de l'estimateur de
Nadaraya-Watson diminue quasiment sans limite quand h se rapproche de
0.

In [ ]:
bandwidths_h = np.logspace(-1.5, 1, 20)
mse_train_h = [
    np.mean((y_train - nadaraya_watson(x_train, y_train, x_train, h)) ** 2)
    for h in bandwidths_h
]

plt.figure(figsize=(6.5, 4.2))
plt.plot(bandwidths_h, mse_train_h, marker="o")
plt.xscale("log")
plt.xlabel("largeur de bande h (échelle log)"); plt.ylabel("MSE (in-sample)")
plt.title("MSE calculée sur les données d'apprentissage elles-mêmes")
plt.tight_layout()
plt.show()


### Exercice 6 (Master)

Ce dernier graphique n'est pas utilisable pour choisir h : quand h tend
vers 0, chaque prédiction se rapproche de l'observation la plus proche
d'elle-même, et la MSE *in-sample* s'effondre sans que le modèle
généralise mieux pour autant. On choisit donc h avec un vrai ensemble de
validation.

1. À partir de `x_data, y_data` (fournis ci-dessous, sinusoïde bruitée à
   150 points), séparez en `(x_fit, y_fit)` / `(x_val, y_val)` avec
   `train_test_split(test_size=0.3, random_state=0)`.
2. Pour chaque largeur de bande de `bandwidths_a_tester`, calculez la
   MSE de validation avec `nadaraya_watson` entraîné implicitement sur
   `(x_fit, y_fit)`.
3. Tracez la MSE de validation en fonction de h (échelle log) et
   identifiez la largeur de bande optimale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def nadaraya_watson(x_train, y_train, x_query, bandwidth):
    x_train = np.asarray(x_train).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)
    x_query = np.asarray(x_query).reshape(-1)
    u = (x_query[:, None] - x_train[None, :]) / bandwidth
    poids = np.exp(-0.5 * u ** 2)
    numerateur = poids @ y_train
    denominateur = poids.sum(axis=1)
    denominateur = np.where(denominateur < 1e-12, 1e-12, denominateur)
    return numerateur / denominateur

rng = np.random.RandomState(1)
x_data = np.sort(rng.uniform(0, 10, size=150))
y_data = np.sin(x_data) + rng.normal(scale=0.3, size=x_data.shape[0])
bandwidths_a_tester = np.logspace(-1.5, 1, 20)

x_fit, x_val, y_fit, y_val = train_test_split(x_data, y_data, test_size=0.3, random_state=0)

mse_validation = []
for h in bandwidths_a_tester:
    y_pred_val = nadaraya_watson(x_fit, y_fit, x_val, h)
    mse_validation.append(np.mean((y_val - y_pred_val) ** 2))
mse_validation = np.array(mse_validation)

h_optimal = bandwidths_a_tester[np.argmin(mse_validation)]

plt.figure(figsize=(6.5, 4.5))
plt.plot(bandwidths_a_tester, mse_validation, marker="o")
plt.xscale("log")
plt.xlabel("largeur de bande h (échelle log)"); plt.ylabel("MSE de validation")
plt.title("Sélection de la largeur de bande par validation")
plt.axvline(h_optimal, color="crimson", linestyle="--", label=f"h optimal ~ {h_optimal:.2f}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Largeur de bande optimale : {h_optimal:.3f}")
print(f"MSE de validation minimale : {mse_validation.min():.4f}")
print("\nLes largeurs de bande extrêmes (très petites ou très grandes) donnent")
print("une MSE de validation plus élevée qu'une largeur intermédiaire : on")
print("retrouve le compromis biais-variance annoncé en cours.")


Le noyau gaussien n'est pas le seul choix possible : un **noyau
uniforme** (ou fenêtre), $K(u)=\mathbb{1}_{[-1,1]}(u)$, attribue un poids
identique à toutes les observations situées à une distance normalisée
inférieure à 1, et un poids nul au-delà.

In [ ]:
def nadaraya_watson_generique(x_train, y_train, x_query, bandwidth, noyau):
    x_train = np.asarray(x_train).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)
    x_query = np.asarray(x_query).reshape(-1)
    u = (x_query[:, None] - x_train[None, :]) / bandwidth
    poids = noyau(u)
    numerateur = poids @ y_train
    denominateur = poids.sum(axis=1)
    denominateur = np.where(denominateur < 1e-12, 1e-12, denominateur)
    return numerateur / denominateur

noyau_gaussien = lambda u: np.exp(-0.5 * u ** 2)
noyau_uniforme = lambda u: (np.abs(u) <= 1).astype(float)

y_gauss = nadaraya_watson_generique(x_train, y_train, x_grid, 0.5, noyau_gaussien)
y_uniforme = nadaraya_watson_generique(x_train, y_train, x_grid, 0.5, noyau_uniforme)

plt.figure(figsize=(7.5, 4.5))
plt.scatter(x_train, y_train, s=20, color="gray", alpha=0.6, label="données")
plt.plot(x_grid, y_gauss, color="crimson", lw=2, label="noyau gaussien")
plt.plot(x_grid, y_uniforme, color="seagreen", lw=2, label="noyau uniforme (fenêtre)")
plt.legend()
plt.title("Noyau gaussien vs noyau uniforme, même largeur de bande h=0.5")
plt.tight_layout()
plt.show()


On note que la courbe obtenue avec le noyau uniforme est moins lisse :
ce noyau attribue un poids identique à toutes les observations dans la
fenêtre puis un poids brutalement nul au-delà, alors que le noyau
gaussien décroît progressivement avec la distance.

### Exercice 7 (Master)

1. Implémentez `nadaraya_watson_uniforme(x_train, y_train, x_query, h)`
   avec le noyau uniforme $K(u)=\mathbb{1}_{[-1,1]}(u)$ (sans passer par
   `nadaraya_watson_generique`).
2. Sur l'échantillon $(1,10),(2,20),(4,30)$, vérifiez que la prédiction
   en $x_0=2$ vaut $15$ pour $h=1.5$ (seules $x_1$ et $x_2$ tombent dans
   la fenêtre).
3. Vérifiez qu'elle vaut $20$ (la moyenne globale de $y$) pour un $h$
   assez grand pour que les 3 observations tombent dans la fenêtre.

In [ ]:
import numpy as np

def nadaraya_watson_uniforme(x_train, y_train, x_query, bandwidth):
    x_train = np.asarray(x_train).reshape(-1)
    y_train = np.asarray(y_train).reshape(-1)
    x_query = np.asarray(x_query).reshape(-1)
    u = (x_query[:, None] - x_train[None, :]) / bandwidth
    poids = (np.abs(u) <= 1).astype(float)
    numerateur = poids @ y_train
    denominateur = poids.sum(axis=1)
    denominateur = np.where(denominateur < 1e-12, 1e-12, denominateur)
    return numerateur / denominateur

x_obs = np.array([1.0, 2.0, 4.0])
y_obs = np.array([10.0, 20.0, 30.0])

pred_h_petit = nadaraya_watson_uniforme(x_obs, y_obs, [2.0], 1.5)[0]
pred_h_grand = nadaraya_watson_uniforme(x_obs, y_obs, [2.0], 10.0)[0]

print(f"Prédiction en x0=2, h=1.5  : {pred_h_petit}")
print(f"Prédiction en x0=2, h=10.0 : {pred_h_grand}")

assert np.isclose(pred_h_petit, 15.0, atol=1e-6)
assert np.isclose(pred_h_grand, 20.0, atol=1e-6)
print("\nOK : on retrouve bien 15 (fenêtre restreinte à x1, x2) et 20 (moyenne")
print("globale de y, une fois les 3 observations dans la fenêtre).")


## Pour la suite

La régression à noyau referme la boucle amorcée avec le k-NN : deux
méthodes qui ne fixent aucune forme fonctionnelle globale, et qui payent
cette flexibilité par la nécessité de garder tout l'échantillon
d'apprentissage en mémoire. Le prochain notebook revient à des modèles
paramétriques, mais construits très différemment : arbres de décision et
réseaux de neurones.